# Prepare NN-duel forecasts with non-overlapping training histories

Run this notebook before **15_nonoverlap_nn_duel_skill_forecasting.ipynb**.
It reads the same complete notebook-11 corpus as notebook 12, preserves its seeded
40/30/30 scenario-group split and forecast contract, and creates a separate dataset
with fewer repeated training intervals. Notebook 12 is not a prerequisite.

**Training histories do not overlap, including across the two aircraft views of a
physical flight.** Each model input has 51 frames spanning five seconds. Including
its backward-acceleration lookback, it uses 56 raw frames. Retained training origins
are at least **60 ticks / 6 seconds** apart. Six-second slots alternate aircraft
views; a stable hash of the flight ID and split seed chooses the first view, without
looking at labels. If a slot is ineligible, it stays empty. Test and validation keep
the original one-second candidate stride, ensuring the same held-out examples as
notebook 12 for the same source and split seed.

For A observing B, inputs contain A's 51 observations, three causal acceleration
features and current own-skill indicators. Targets are B's selected skill at exact
+1/+2/+5 seconds. Missing labels remain independently masked. Both views of every
flight stay in one partition. Scaling uses only retained training-history frames;
own-skill indicators remain binary. Future target timestamps may fall in a later
history; non-overlap removes shared raw input intervals, not all within-flight
statistical dependence. It does not create more independent flights.

**Run All downloads required Git LFS payloads and prepares data; it does not collect
flights or train models.** Install `.[ml]` and a Jupyter/IPython kernel first.
The 10,000-flight archived release is about 10.3 GB; this separate dataset keeps the
full memory-mapped frame array (about 13.7 GiB), plus a smaller window index.
Allow at least 30 GB for download and preparation; existing prepared arrays consume
additional space. Frames are streamed without expanding windows. Every executable
cell is timed. Preparation and training use their own artifact root and receipt.

For a Windows PyArrow import failure, see the [runtime repair instructions](../docs/nn_duel_skill_forecasting.md#windows-pyarrow-import-failure).


In [1]:
%%time
from pathlib import Path
import os
import sys

root_override = os.environ.get("BVR_NN_ROOT")
candidates = ([Path(root_override).expanduser()] if root_override else
              [Path.cwd(), *Path.cwd().parents])
ROOT = next((p.resolve() for p in candidates
             if (p / "bvr_behavior_prediction/nn_duel_skill_forecasting.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set BVR_NN_ROOT to the full Trajectory_Classification checkout path.")
sys.path.insert(0, str(ROOT))
from bvr_behavior_prediction.notebook_runtime import require_pyarrow
require_pyarrow()  # Report the kernel environment and native-library repair steps.
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from bvr_behavior_prediction import nn_duel_skill_forecasting as duel_data
from bvr_behavior_prediction.paired_skill_forecasting import (
    HORIZONS, HISTORY_STEPS, RAW_HISTORY_STEPS, SPLITS, read_json, write_json, sha256,
)
EXPERIMENT_ROOT = ROOT / "artifacts/nn_duel_nonoverlap_forecasting"
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

SOURCE_OVERRIDE = os.environ.get("BVR_NN_FORECAST_SOURCE")  # Full path, or repository-relative.
SOURCE = duel_data.discover_corpus(ROOT, SOURCE_OVERRIDE)
SOURCE_MANIFEST = read_json(SOURCE / "manifest.json")
SOURCE_DIGEST = sha256(SOURCE / "manifest.json")
PREPARATION_DIGEST = sha256(duel_data.__file__)
FEATURE_HELPERS_DIGEST = sha256(ROOT / "bvr_behavior_prediction/paired_skill_forecasting.py")
DATA_DEFAULT = (EXPERIMENT_ROOT / "data" /
                f"{SOURCE.name}_{SOURCE_DIGEST[:12]}_{PREPARATION_DIGEST[:8]}_{FEATURE_HELPERS_DIGEST[:8]}")
DATA = Path(os.environ.get("BVR_NN_NONOVERLAP_DATA", str(DATA_DEFAULT))).expanduser().resolve()
DOWNLOAD_LFS = os.environ.get("BVR_NN_FORECAST_DOWNLOAD", "1") == "1"
GIT_EXECUTABLE = None  # e.g. r"C:\Program Files\Git\cmd\git.exe" on Windows.
SPLIT_SEED = 1729
print("Source:", SOURCE)
print("Complete:", SOURCE_MANIFEST["complete"], "| physical flights:",
      SOURCE_MANIFEST["physical_flights"], "| perspectives:", SOURCE_MANIFEST["perspective_episodes"])
print("Source manifest SHA-256:", SOURCE_DIGEST)
print("Prepared output:", DATA)


Source: C:\Users\theon\Trajectory_Classification\shared\corpora\nn-duels-20k-v1\eee3474c5493
Complete: True | physical flights: 10000 | perspectives: 20000
Source manifest SHA-256: 5b951842316d54e51f56d540cb0d8452b8aac9d646361504abdf7dcb9dbf0509
Prepared output: C:\Users\theon\Trajectory_Classification\artifacts\nn_duel_nonoverlap_forecasting\data\eee3474c5493_5b951842316d_0a5d2704_7661ce18
CPU times: total: 1.59 s
Wall time: 1.62 s


## Alignment and forecasting contract

* Histories contain **51 chronological frames at 10 Hz**, spanning five seconds.
  Five additional past frames support backward 0.5-second acceleration. The first
  forecast origin is 6 seconds. Training uses six-second slots across both views,
  alternating the observer. Every training raw-history interval is disjoint from
  every other retained interval in the same flight. Test/validation use one-second
  stride. No input history crosses a flight.
* For observer A, its own A decision comes from **B's `target_decisions` file**.
  Actual aircraft IDs and episode keys are checked before alignment. A decision
  at tick 60 applies at ticks 60–69; the decision at 70 cannot affect earlier inputs.
  Missing own decision intervals fail preparation rather than being filled indefinitely.
  This is the pilot's selected skill known when issuing guidance at the frame timestamp.
* Future opponent labels require an **exact recorded decision**. Missing +1/+2/+5
  labels are masked independently with `-1`; there is no extrapolation beyond termination.
  Opponent decisions are labels only. Pilot/checkpoint IDs and outcomes are metadata.
* **40% train / 30% test / 30% validation**, grouped by `scenario_group`, keep
  both perspectives of every physical flight together. Seeded range strata preserve
  starting-range balance while balancing per-skill/horizon episode and window support.
  Continuous NN-duel ranges require strata in place of 09b's discrete range quotas.
  Splits are balanced on the original eligible candidate set before training thinning,
  matching notebook 12. Rare skills may lose support after thinning; inspect the
  retained per-class/horizon support and sampling audits.
* Scaling is fitted on unique frames used by **retained training histories only**. The
  appended skill indicators stay binary (mean 0, scale 1). No future labels or
  validation frames enter normalization. Earlier corpus files and 09b caches are preserved.


In [2]:
%%time
if DOWNLOAD_LFS:
    duel_data.download_corpus(ROOT, SOURCE, git_executable=GIT_EXECUTABLE)

analysis = duel_data.analyze_corpus(ROOT, SOURCE)
support = duel_data.skill_support_table(analysis)
print(f"Surveyed {len(analysis['episodes']):,} perspectives before assigning splits.")
display(analysis["episodes"].groupby(["observer_id", "family"]).agg(
    perspectives=("episode_id", "size"), mean_duration_s=("duration_s", "mean")))
with pd.option_context("display.max_rows", None):
    display(support)
print("Zero-window catalogue labels:", sorted(set(analysis["classes"]) -
      set(support.loc[support.windows.gt(0), "skill"])))


Read part-00000: 200 perspective episodes
Read part-00001: 200 perspective episodes
Read part-00002: 200 perspective episodes
Read part-00003: 200 perspective episodes
Read part-00004: 200 perspective episodes
Read part-00005: 200 perspective episodes
Read part-00006: 200 perspective episodes
Read part-00007: 200 perspective episodes
Read part-00008: 200 perspective episodes
Read part-00009: 200 perspective episodes
Read part-00010: 200 perspective episodes
Read part-00011: 200 perspective episodes
Read part-00012: 200 perspective episodes
Read part-00013: 200 perspective episodes
Read part-00014: 200 perspective episodes
Read part-00015: 200 perspective episodes
Read part-00016: 200 perspective episodes
Read part-00017: 200 perspective episodes
Read part-00018: 200 perspective episodes
Read part-00019: 200 perspective episodes
Read part-00020: 200 perspective episodes
Read part-00021: 200 perspective episodes
Read part-00022: 200 perspective episodes
Read part-00023: 200 perspective e

perspectives  mean_duration_s
observer_id family                                         
A01         close_merge               1000         110.4747
            cold_recommit             1000         277.7224
            crossing_left             1000         237.1058
            crossing_right            1000         214.5859
            defensive_beam            1000         103.1956
            high_low                  1000         246.7570
            low_energy_climb          1000         261.1450
            mixed_aspect              1000         233.6012
            offset_support            1000         183.1533
            tail_chase                1000         241.0621
B01         close_merge               1000         110.4747
            cold_recommit             1000         277.7224
            crossing_left             1000         237.1058
            crossing_right            1000         214.5859
            defensive_beam            1000         103.1956
            high_low                  1000         246.7570
            low_energy_climb          1000         261.1450
            mixed_aspect              1000         233.6012
            offset_support            1000         183.1533
            tail_chase                1000         241.0621

,split,horizon_s,skill,episodes,windows,scenario_groups
0,all,1,abort_support,379,4970,375
1,all,1,accelerate_to_speed,2,6,2
2,all,1,beam_missile_left,149,846,148
3,all,1,beam_missile_right,478,3587,475
4,all,1,beam_target_left,5981,163502,4946
5,all,1,beam_target_right,4310,75138,3712
6,all,1,climb_to_altitude,17,64,17
7,all,1,commit,2,7,2
8,all,1,crank_target_left,5806,57298,4767
9,all,1,crank_target_right,3554,126340,3241


Zero-window catalogue labels: []
CPU times: total: 7min 35s
Wall time: 7min 28s


In [3]:
%%time
if (DATA / "manifest.json").exists():
    manifest = read_json(DATA / "manifest.json")
    if SOURCE_DIGEST not in manifest["source_manifests"].values():
        raise ValueError("The existing dataset belongs to another source; choose a new DATA path.")
    if manifest["source_code_sha256"] != PREPARATION_DIGEST:
        raise ValueError("Preparation code changed; choose a fresh DATA path.")
    if manifest["feature_helpers_sha256"] != FEATURE_HELPERS_DIGEST:
        raise ValueError("Feature helper code changed; choose a fresh DATA path.")
    if manifest["split_seed"] != SPLIT_SEED:
        raise ValueError("Split seed changed; choose a fresh DATA path.")
    print("Reusing the verified prepared dataset:", DATA)
else:
    manifest = duel_data.prepare_dataset(ROOT, DATA, source=SOURCE,
                                         split_seed=SPLIT_SEED, analysis=analysis,
                                         nonoverlapping_train=True)
data = duel_data.NonoverlappingNNDuelData(DATA)  # Verify stored file hashes and the input/target contract.
assert data.features.shape[1] == 54 + len(data.classes)
assert data.manifest["features"] == manifest["features"]
display(pd.DataFrame(manifest["counts"]).T)
display(pd.Series({"frames": len(data.features), "features_per_frame": data.features.shape[1],
                   "feature_array_GiB": data.features.nbytes / 2**30,
                   "history_steps": HISTORY_STEPS, "horizons_s": HORIZONS}))

# The loader verifies disjoint raw intervals across both views, not just per observer.
display(pd.Series(duel_data.nonoverlap_audit(data.samples)))
sampling = pd.read_csv(DATA / "window_sampling_audit.csv")
display(sampling.groupby("split")[["eligible_windows", "retained_windows", "dropped_windows"]].sum())

# Show physical-flight intervals across both observer views on the same time axis.
training_intervals = data.samples.loc[data.samples.split.eq("train"),
    ["episode_id", "observer_id", "step"]].sort_values(["episode_id", "step"])
training_intervals = training_intervals.assign(
    raw_start_tick=training_intervals.step - RAW_HISTORY_STEPS + 1,
    model_start_tick=training_intervals.step - HISTORY_STEPS + 1,
    raw_end_tick=training_intervals.step)
display(training_intervals.head(12))


Read part-00000: 200 perspective episodes
Read part-00001: 200 perspective episodes
Read part-00002: 200 perspective episodes
Read part-00003: 200 perspective episodes
Read part-00004: 200 perspective episodes
Read part-00005: 200 perspective episodes
Read part-00006: 200 perspective episodes
Read part-00007: 200 perspective episodes
Read part-00008: 200 perspective episodes
Read part-00009: 200 perspective episodes
Read part-00010: 200 perspective episodes
Read part-00011: 200 perspective episodes
Read part-00012: 200 perspective episodes
Read part-00013: 200 perspective episodes
Read part-00014: 200 perspective episodes
Read part-00015: 200 perspective episodes
Read part-00016: 200 perspective episodes
Read part-00017: 200 perspective episodes
Read part-00018: 200 perspective episodes
Read part-00019: 200 perspective episodes
Read part-00020: 200 perspective episodes
Read part-00021: 200 perspective episodes
Read part-00022: 200 perspective episodes
Read part-00023: 200 perspective e

,groups,episodes,windows,labels_by_horizon
train,4000,8000,137001,"[137001, 136791, 136135]"
test,3000,6000,1224086,"[1224086, 1218086, 1200100]"
validation,3000,6000,1224414,"[1224414, 1218414, 1200422]"


frames                 42196060
features_per_frame           87
feature_array_GiB     13.675754
history_steps                51
horizons_s            (1, 2, 5)
dtype: object

training_windows                137001
minimum_origin_spacing_steps        60
raw_history_steps                   56
overlapping_pairs                    0
dtype: int64

,eligible_windows,retained_windows,dropped_windows
split,,,
test,1224086,1224086,0
train,1632134,137001,1495133
validation,1224414,1224414,0


,episode_id,observer_id,step,raw_start_tick,model_start_tick,raw_end_tick
0,nn_1883299364_00000,A01,60,5,10,60
13705,nn_1883299364_00000,B01,120,65,70,120
1,nn_1883299364_00000,A01,180,125,130,180
13706,nn_1883299364_00000,B01,240,185,190,240
2,nn_1883299364_00000,A01,300,245,250,300
13707,nn_1883299364_00000,B01,360,305,310,360
3,nn_1883299364_00000,A01,420,365,370,420
13708,nn_1883299364_00000,B01,480,425,430,480
4,nn_1883299364_00000,A01,540,485,490,540
13709,nn_1883299364_00000,B01,600,545,550,600


CPU times: total: 8min 26s
Wall time: 8min 29s


## Inspect the split and the appended inputs

The support tables retain zero-support classes. A missing class in validation is
visible here and is not repaired by changing the held-out partition after training.
The example below checks a normalized history from a training sample: its appended
indicators still identify the observing aircraft's own skill at every frame.

`skill_support_before_split.csv` describes all eligible candidates;
`skill_support_by_split.csv` describes the actual retained labels after thinning.
`window_sampling_audit.csv` records eligible/retained/dropped windows per flight view,
and `nonoverlap_audit.json` records verified raw-history disjointness.


In [4]:
%%time
assignments = pd.read_csv(DATA / "split_assignments.csv")
assert assignments.groupby("scenario_group").split.nunique().eq(1).all()
assert assignments.groupby("episode_id").observer_id.nunique().eq(2).all()
assert assignments.groupby("episode_id").split.nunique().eq(1).all()
display(assignments.groupby(["split", "observer_id"]).size().unstack(fill_value=0))
split_support = pd.read_csv(DATA / "skill_support_by_split.csv")
with pd.option_context("display.max_rows", None):
    display(split_support.pivot(index=["horizon_s", "skill"], columns="split",
                                values=["episodes", "windows", "scenario_groups"]))
sample_id = int(data.indices["train"][0])
x, y = data.batch(np.asarray([sample_id]), "cpu")
own = x[0, :, 54:].numpy()
assert np.isin(own, [0, 1]).all() and np.all(own.sum(axis=1) == 1)
assert np.all(data.mean[54:] == 0) and np.all(data.std[54:] == 1)
sample = data.samples.iloc[sample_id]
display(sample)
display(pd.DataFrame({
    "time_s": np.arange(sample.step - HISTORY_STEPS + 1, sample.step + 1) * .1,
    "observer_current_skill": np.asarray(data.classes)[own.argmax(axis=1)]}))
display(pd.Series({f"opponent_at_+{h}s": data.classes[int(t)] if t >= 0 else "MASKED"
                   for h, t in zip(HORIZONS, y[0].numpy())}))

# Written only after preparation and verification finish. Notebook 15 ingests this exact dataset.
write_json(EXPERIMENT_ROOT / "latest_dataset.json", {
    "directory": str(DATA), "manifest_sha256": sha256(DATA / "manifest.json"),
    "source": str(SOURCE), "source_manifest_sha256": SOURCE_DIGEST})
print("Ready for notebook 15:", DATA)


observer_id,A01,B01
split,,
test,3000,3000
train,4000,4000
validation,3000,3000


episodes                  windows         \
split                             test train validation    test  train   
horizon_s skill                                                          
1         abort_support            114    90        113    1488    173   
          accelerate_to_speed        1     1          0       2      1   
          beam_missile_left         44    31         45     253     34   
          beam_missile_right       144   116        143    1073    132   
          beam_target_left        1794  1755       1795   49072   5543   
          beam_target_right       1292  1219       1293   22525   2644   
          climb_to_altitude          5     1          5      20      1   
          commit                     1     0          0       2      0   
          crank_target_left       1741  1552       1743   17193   2179   
          crank_target_right      1067  1076       1065   37913   4229   
          decelerate_to_speed      262   269        262    3324    409   
          defensive_reversal        39    32         38     455     54   
          descend_to_altitude      279   253        279    3126    382   
          dive_defense               2     1          3       9      1   
          drag_missile               7     5          7      39      5   
          extend                  1168  1328       1166  100323  11173   
          lag_pursuit             1916  2303       1916  194366  21647   
          last_ditch_break           4     2          3      17      5   
          launch                   226   179        225    2710    321   
          lead_pursuit             734   700        734   19044   2150   
          lock_target               12     7         12      60      7   
          maintain_heading          48    33         48     309     39   
          notch_left               225   161        225    1479    183   
          notch_right              104    74        104     635     78   
          preemptive_crank           7     5          7      40      6   
          pursue_target            841   945        842   54139   6037   
          recommit                2616  3213       2617  347921  38551   
          search                     9     6          9      52      7   
          secondary_shot          2429  2527       2429  104198  11696   
          short_range_attack       450   409        449    4443    560   
          support_missile           97    76         97     981    116   
          turn_cold               2181  2685       2182  256315  28569   
          turn_to_heading           75    56         75     560     69   
2         abort_support            114    91        113    1477    169   
          accelerate_to_speed        1     1          0       2      1   
          beam_missile_left         42    29         43     248     29   
          beam_missile_right       141   116        142    1015    130   
          beam_target_left        1755  1717       1755   48555   5524   
          beam_target_right       1267  1197       1269   22040   2595   
          climb_to_altitude          5     3          5      19      3   
          commit                     1     1          0       2      1   
          crank_target_left       1679  1504       1678   16215   2099   
          crank_target_right      1063  1073       1061   37773   4220   
          decelerate_to_speed      258   263        259    3202    402   
          defensive_reversal        37    31         37     434     52   
          descend_to_altitude      272   242        274    2988    360   
          dive_defense               2     0          3       9      0   
          drag_missile               7     4          7      39      4   
          extend                  1162  1331       1162  100121  11164   
          lag_pursuit             1907  2308       1910  193702  21660   
          last_ditch_break           4     2          3      17      4   
          launch                  

episode_id        nn_1883299364_00000
observer_id                       A01
target_id                         B01
pilot_id                    pilot_004
scenario_group    nn_1883299364_00000
split                           train
step                               60
time_s                            6.0
Name: 0, dtype: object

,time_s,observer_current_skill
0,1.0,descend_to_altitude
1,1.1,descend_to_altitude
2,1.2,descend_to_altitude
3,1.3,descend_to_altitude
4,1.4,descend_to_altitude
5,1.5,descend_to_altitude
6,1.6,descend_to_altitude
7,1.7,descend_to_altitude
8,1.8,descend_to_altitude
9,1.9,descend_to_altitude


opponent_at_+1s    turn_cold
opponent_at_+2s    turn_cold
opponent_at_+5s    turn_cold
dtype: str

Ready for notebook 15: C:\Users\theon\Trajectory_Classification\artifacts\nn_duel_nonoverlap_forecasting\data\eee3474c5493_5b951842316d_0a5d2704_7661ce18
CPU times: total: 125 ms
Wall time: 93 ms
